# 최종 습도 이력 모형 검토

독립적인 고정 길이 과거 창의 합산을 사용합니다. 지역·겨울 동시 제외와 두 겨울 지역 제외를 구분합니다. 반복 개발 자료로 공식 점수·새 독립 표본이 아닙니다.

In [ ]:
from pathlib import Path
import sys,json
import pandas as pd
import numpy as np
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'config/weather_humidity_stable_20261101.json').is_file())
sys.path.insert(0,str(ROOT))
from src.data_source import sha256_file as sha
plan=json.loads((ROOT/'config/weather_humidity_stable_20261101.json').read_text('utf-8'))
out=ROOT/plan['output']
summary=json.loads((out/'summary.json').read_text('utf-8'))
audit=json.loads((out/'independent_verification.json').read_text('utf-8'))
assert audit['status']=='passed' and audit['summary_sha256']==sha(out/'summary.json')
assert summary['primary_success'] and len(summary['fits'])==12
for name,digest in summary['files'].items(): assert sha(out/name)==digest,name
print('12개 신규 적합·파일·검산 확인')

In [ ]:
rows=[]
for mode,r in summary['reports'].items():
    for method,values in r['metrics'].items():
        rows.append(dict(validation=mode,method=method,**values['pooled']))
display(pd.DataFrame(rows))
display(pd.DataFrame({mode:r['gates'] for mode,r in summary['reports'].items()}))
print('독립 문맥 불변 특성 값:',audit['prefix_bit_exact_feature_values'])

In [ ]:
import joblib
from threadpoolctl import threadpool_limits
from src.road_weather_transfer import baselines
f=pd.read_parquet(ROOT/plan['input']).sort_values(['station','time']).reset_index(drop=True)
x=pd.read_parquet(out/'features.parquet')
a,b=baselines(f,'relative_humidity')
query=f.group.eq('honam_baekyang') & f.relative_humidity.notna() & np.isfinite(a) & np.isfinite(b)
model=joblib.load(out/'spatial/honam_baekyang.joblib')
with threadpool_limits(limits=2): actual=np.clip(model.predict(x.loc[query])+b[query],0,100)
np.testing.assert_array_equal(actual,np.load(out/'spatial/honam_baekyang.npy',allow_pickle=False))
print('대표 지역 모형 실제 재추론 일치:',len(actual),'행')

In [ ]:
from IPython.display import Image,display
display(Image(filename=str(ROOT/'docs/images/weather_humidity_stable_20261101.png')))
display(Image(filename=str(ROOT/'docs/images/weather_adopted_20261101.png')))

실제 최종 모형의 전체 재학습·432행 추론 검증은 별도 실행 묶음 `weather_history_supplement_20261101_v2`의 노트북을 사용합니다. 첫 구현의 문맥 의존 오류와 카운터 수정도 보존해 추적할 수 있습니다.